# 7장 프레임워크 및 드라이버 계층: 외부 인터페이스

파이썬으로 구현하는 클린 아키텍처 - 7장 프레임워크 및 드라이버 계층: 외부 인터페이스 코드 예제

> **[노트북 참고]** 아래 셀은 노트북 환경에서 `TodoApp` 코드를 import할 수 있도록 경로를 설정합니다. 반드시 첫 번째로 실행해 주세요.

In [ ]:
# ============================================================
# [추가] 노트북 환경 설정
# TodoApp 패키지를 import하기 위한 경로 설정 (Colab/로컬 환경 자동 감지)
# 반드시 첫 번째로 실행해 주세요.
# ============================================================
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('/content/repo'):
        !git clone https://github.com/songys/Clean-Architecture-with-Python.git /content/repo
    TODOAPP_PATH = '/content/repo/Chapter_7/TodoApp'
else:
    TODOAPP_PATH = os.path.join(os.getcwd(), 'TodoApp')

if TODOAPP_PATH not in sys.path:
    sys.path.insert(0, TODOAPP_PATH)

## 개요

이번 장에서는 클린 아키텍처가 핵심 비즈니스 로직을 깨끗하고 보호된 상태로 유지하면서 외부 프레임워크, 데이터베이스, 서비스와 통합하는 방법을 ﻿살펴본다.

이 장에서 다루는 주요 주제:
* 프레임워크 및 드라이버 계층의 이해
* UI 프레임워크 어댑터 생성
* 구성 요소 구성 및 경계 설정

### 00_framework_example.py

## 프레임워크 vs 드라이버

**프레임워크**(Flask 등)는 컨트롤러+프레젠터의 전체 어댑터 스택이 필요하고, **드라이버**(SQLite 리포지토리 등)는 포트 인터페이스와 구체적 구현만으로 충분하다.

> **[노트북 참고]** 아래 셀은 Flask 프레임워크의 동작을 시연하기 위한 코드입니다. 노트북 환경에서 실행 가능하도록 Flask 스텁과 필요한 import가 추가되었습니다.

In [ ]:
# 프레임워크 vs 드라이버 비교 예제
# 프레임워크: 전체 인터페이스 어댑터 스택(컨트롤러+프레젠터) 필요
# 드라이버: 인터페이스와 구현만으로 충분
# [추가] Flask 스텁 및 필요한 import — 노트북 환경에서 실행 가능하도록 수정
from types import SimpleNamespace
from todo_app.domain.entities.task import Task
from todo_app.application.repositories.task_repository import TaskRepository

# [추가] Flask가 설치되어 있지 않아도 실행되도록 스텁 생성
try:
    from flask import Flask, request
    app = Flask(__name__)
except ImportError:
    class _StubFlask:
        def route(self, *args, **kwargs):
            def decorator(f):
                return f
            return decorator
    app = _StubFlask()
    request = SimpleNamespace(json={"title": "", "description": ""})

# [추가] 시연용 컨트롤러/프레젠터 스텁
task_controller = SimpleNamespace(handle_create=lambda **kw: {"status": "stub"})
task_presenter = SimpleNamespace(present=lambda r: r)

# 프레임워크 예제 - 컨트롤러와 프레젠터를 모두 거치는 전체 어댑터 스택
@app.route("/tasks", methods=["POST"])
def create_task():
    """프레임워크는 전체 인터페이스 어댑터 스택이 필요"""
    result = task_controller.handle_create(  # 6장의 컨트롤러
        title=request.json["title"], description=request.json["description"]
    )
    return task_presenter.present(result)  # 6장의 프레젠터


# 드라이버 예제 - 인터페이스 + 구현만으로 충분한 단순 구조
class SQLiteTaskRepository(TaskRepository):  # 5장의 인터페이스
    """드라이버는 기본적인 인터페이스만 구현"""

    def save(self, task: Task) -> None:
        self.connection.execute(
            "INSERT INTO tasks (id, title) VALUES (?, ?)", (str(task.id), task.title)
        )

    # [추가] 추상 메서드 최소 구현 — TaskRepository 인터페이스 충족
    def get(self, task_id):
        pass
    def delete(self, task_id):
        pass
    def find_by_project(self, project_id):
        return []
    def get_active_tasks(self):
        return []

### 00_click_cli_structure.py

## CLI 프레임워크 어댑터

`ClickCli` 어댑터의 핵심 원칙:
- **의존성 주입**: 생성자를 통해 `Application` 인스턴스를 주입받아 의존성 규칙 준수
- 어댑터 내에서 애플리케이션 구성 요소를 직접 인스턴스화하지 않음

> **[노트북 참고]** 아래 셀은 노트북 환경에서 실행 가능하도록 `Application` 클래스와 `click` 라이브러리 import가 추가되었습니다.

In [ ]:
# CLI 프레임워크 어댑터(ClickCli) - Application 컨테이너를 주입받아 CLI 인터페이스를 제공
# 의존성 주입: 생성자를 통해 Application 인스턴스를 받아 의존성 규칙 준수
# [추가] 필요한 import
try:
    from todo_app.infrastructure.configuration.container import Application
except (ImportError, ModuleNotFoundError):
    class Application:  # type: ignore[no-redef]
        """[스텁] 외부 패키지 미설치 시 사용되는 Application 스텁"""
        pass
try:
    import click
except ImportError:
    from types import SimpleNamespace as _NS
    class _ClickStub:
        def echo(self, *a, **kw): pass
        def secho(self, *a, **kw): pass
        def clear(self): pass
        def prompt(self, *a, **kw): return ""
        def confirm(self, *a, **kw): return False
        def pause(self): pass
    click = _ClickStub()

class ClickCli:
    def __init__(self, app: Application):
        # 의존성 주입: 외부에서 조립된 Application 컨테이너를 전달받음
        self.app = app
        self.current_projects = []  # 표시용 프로젝트 캐시 목록

    # CLI 애플리케이션의 메인 루프 - 사용자 상호작용 처리
    def run(self) -> int:
        """Click CLI 애플리케이션 실행 진입점"""
        try:
            while True:
                self._display_projects()
                self._handle_selection()
        except KeyboardInterrupt:
            click.echo("\n안녕히 가세요!", err=True)
            return 0

    # 작업 상세 정보 표시 - 컨트롤러를 통해 유스케이스 호출
    def _display_task_menu(self, task_id: str) -> None:
        """작업 메뉴를 표시와 처리"""
        result = self.app.task_controller.handle_get(task_id)
        if not result.is_success:
            click.secho(result.error.message, fg="red", err=True)
            return
        # 뷰 모델의 미리 형식화된 필드를 단순 출력 (험블 뷰 패턴)
        task = result.success
        click.clear()
        click.echo("\n작업 상세 정보")
        click.echo("=" * 40)
        click.echo(f"제목:       {task.title}")
        click.echo(f"설명:       {task.description}")
        click.echo(f"상태:       {task.status_display}")
        click.echo(f"우선순위:   {task.priority_display}")

    def _handle_selection(self) -> None:
        """프로젝트/작업 선택을 처리"""
        selection = (
            click.prompt(
                "\n프로젝트 또는 작업을 선택하세요 (예: '1' 또는 '1.a')", type=str, show_default=False
            )
            .strip()
            .lower()
        )
        if selection == "np":
            self._create_new_project()
            return
        try:
            if "." in selection:
                project_num, task_letter = selection.split(".")
                self._handle_task_selection(int(project_num), task_letter)
            else:
                self._handle_project_selection(int(selection))
        except (ValueError, IndexError):
            click.secho(
                "잘못 선택했습니다. 프로젝트는 '1', 작업은 '1.a'를 사용하세요.",
                fg="red",
                err=True,
            )

    def _create_new_project(self) -> None:
        """새 프로젝트 생성 - 컨트롤러를 통한 유스케이스 호출"""
        name = click.prompt("프로젝트 이름", type=str)
        description = click.prompt("설명 (선택사항)", type=str, default="")
        result = self.app.project_controller.handle_create(name, description)
        if not result.is_success:
            click.secho(result.error.message, fg="red", err=True)

    # ... 추가 메서드

### 02_inbox_task_project_relation.py

## 작업-프로젝트 관계와 Inbox 패턴

도메인 내에서 작업이 자연스럽게 프로젝트에 속한다는 점을 코드에 반영하기 위해 필요한 주요 변경 사항을 ﻿살펴본다. 구현은 도메인 계층에서 시작하여 외부로 확장하며, 프로젝트를 지정하지 않는 작업을 담아둘 실용적 메커니즘으로 Inbox 프로젝트를 활용한다.

> **[노트북 참고]** 아래 셀은 노트북 환경에서 실행 가능하도록 `Enum`, `Entity` 등 필요한 import가 추가되었습니다.

In [ ]:
# 작업-프로젝트 관계와 Inbox 패턴 - 도메인 계층의 변경
# 모든 작업이 반드시 프로젝트에 소속되도록 하되, 미분류 작업을 위한 Inbox 프로젝트 제공
from dataclasses import dataclass, field
from uuid import UUID
from enum import Enum  # [추가] Enum import
from todo_app.domain.entities.entity import Entity  # [추가] Entity import


# 프로젝트 유형 열거형 - 일반 프로젝트와 Inbox 프로젝트 구분
class ProjectType(Enum):
    REGULAR = "REGULAR"  # 사용자가 생성한 일반 프로젝트
    INBOX = "INBOX"      # 미분류 작업을 담는 기본 프로젝트


# Project 엔터티 - 프로젝트 유형과 Inbox 팩토리 메서드 추가
@dataclass
class Project(Entity):
    name: str
    description: str = ""
    project_type: ProjectType = field(default=ProjectType.REGULAR)

    # 팩토리 메서드: Inbox 프로젝트 생성 - 미분류 작업의 기본 소속처
    @classmethod
    def create_inbox(cls) -> "Project":
        return cls(
            name="INBOX",
            description="할당되지 않은 작업을 위한 기본 프로젝트",
            project_type=ProjectType.INBOX,
        )


# Task 엔터티 변경 - project_id가 필수 필드로 변경 (더 이상 선택사항이 아님)
@dataclass
class Task(Entity):
    title: str
    description: str
    project_id: UUID  # 더 이상 선택사항이 아님 - 모든 작업은 프로젝트에 소속

### 03_inbox_application_layer.py

## 애플리케이션 계층의 Inbox 처리

`ProjectRepository`에 Inbox 전용 조회 메서드가 추가되고, `CreateTaskUseCase`는 프로젝트 미지정 시 Inbox에 자동 할당한다. 비즈니스 규칙을 유스케이스 한 곳에서 관리하여 일관성을 유지한다.

> **[노트북 참고]** 아래 셀은 노트북 환경에서 실행 가능하도록 `ABC`, `abstractmethod`, `TaskRepository` 등 필요한 import가 추가되었습니다.

In [ ]:
# 애플리케이션 계층의 Inbox 처리 - 리포지토리와 유스케이스 업데이트
# 프로젝트 미지정 시 Inbox 프로젝트에 자동 할당하는 비즈니스 규칙
from abc import ABC, abstractmethod  # [추가]
from dataclasses import dataclass  # [추가]
from todo_app.application.repositories.task_repository import TaskRepository  # [추가]
from todo_app.application.common.result import Result  # [추가]
from todo_app.application.dtos.task_dtos import CreateTaskRequest  # [추가]
from todo_app.domain.entities.project import Project  # [추가]

# ProjectRepository에 Inbox 전용 조회 메서드 추가
class ProjectRepository(ABC):
    @abstractmethod
    def get_inbox(self) -> Project:
        """INBOX 프로젝트 조회 - Inbox 패턴의 핵심 인터페이스"""
        pass

# CreateTaskUseCase - 프로젝트 미지정 시 Inbox 자동 할당 로직
@dataclass
class CreateTaskUseCase:
    task_repository: TaskRepository
    project_repository: ProjectRepository  # Inbox 조회를 위한 의존성 추가

    def execute(self, request: CreateTaskRequest) -> Result:
        try:
            params = request.to_execution_params()
            project_id = params.get("project_id")
            # 비즈니스 규칙: 프로젝트 미지정 시 Inbox 프로젝트에 자동 할당
            if not project_id:
                project_id = self.project_repository.get_inbox().id
            # ... 나머지 구현
        except Exception:  # [수정] bare except 방지
            pass

### 04_inbox_create_task.py

## 간소화된 작업 생성 (Inbox 패턴 적용 후)

Inbox 패턴 덕분에 CLI 어댑터가 크게 간소화된 모습이다. 비즈니스 규칙은 유스 케이스에서 처리한다.

> **[노트북 참고]** 아래 셀은 `ClickCli` 클래스의 메서드를 보여주는 예제입니다. 노트북에서 독립 실행 가능하도록 `click` import와 함수로 감싸는 형태로 수정되었습니다.

In [ ]:
# [보완] 아래 코드는 ClickCli 클래스 메서드의 발췌입니다.
# 노트북에서 구문 오류 없이 정의되도록 독립 함수 형태로 표현합니다.
try:
    import click
except ImportError:
    pass

def _create_task(self) -> None:
    """작업 생성 명령 처리"""
    title = click.prompt("작업 제목", type=str)
    description = click.prompt("설명", type=str)

    # 프로젝트 선택 가능 - 기본값은 Inbox
    project_id = None  # [수정] 변수 초기화 추가
    if click.confirm("특정 프로젝트에 할당하시겠습니까?", default=False):
        project_id = self._select_project()

    # 유스 케이스에서 Inbox 처리
    result = self.app.task_controller.handle_create(
        title=title, description=description, project_id=project_id
    )

### 05_in_mem_and_file_task_repo.py

## 인메모리/파일 리포지토리 구현

동일한 `TaskRepository` 인터페이스를 인메모리(딕셔너리)와 파일(JSON) 두 가지 방식으로 구현한다. 인메모리 구현은 가볍고 빠르므로 테스트에 이상적이며, 8장에서 더 자세히 다룬다.

> **[노트북 참고]** 아래 셀은 노트북 환경에서 실행 가능하도록 `Task`, `TaskNotFoundError` 등 필요한 import가 추가되었습니다.

In [ ]:
# 인메모리/파일 리포지토리 구현 - 동일한 인터페이스의 서로 다른 구현체
# 저장 방식이 달라도 유스케이스는 동일하게 작동 (의존성 역전 원칙)
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Dict
from uuid import UUID
from todo_app.domain.entities.task import Task  # [추가]
from todo_app.domain.exceptions import TaskNotFoundError  # [추가]


# 리포지토리 인터페이스 (애플리케이션 계층의 포트)
class TaskRepository(ABC):
    """Task 엔터티 저장을 위한 리포지토리 인터페이스"""

    @abstractmethod
    def get(self, task_id: UUID) -> Task:
        """ID로 작업을 조회"""
        pass

    @abstractmethod
    def save(self, task: Task) -> None:
        """작업을 리포지토리에 저장"""
        pass

    # ... 나머지 인터페이스 메서드


# 인메모리 리포지토리 - 딕셔너리 기반의 경량 구현 (테스트에 이상적)
class InMemoryTaskRepository(TaskRepository):
    """TaskRepository의 인메모리 구현"""

    def __init__(self) -> None:
        self._tasks: Dict[UUID, Task] = {}  # UUID → Task 매핑 딕셔너리

    def get(self, task_id: UUID) -> Task:
        """ID로 작업 조회 - 딕셔너리에서 직접 검색"""
        if task := self._tasks.get(task_id):
            return task
        raise TaskNotFoundError(task_id)

    def save(self, task: Task) -> None:
        """작업 저장 - 딕셔너리에 UUID 키로 저장"""
        self._tasks[task.id] = task

    # 추가 인터페이스 메서드 구현


# 파일 기반 리포지토리 - JSON 파일을 사용한 영속성 구현
class FileTaskRepository(TaskRepository):
    """TaskRepository의 JSON 파일 기반 구현."""

    def __init__(self, data_dir: Path):
        self.tasks_file = data_dir / "tasks.json"
        self._ensure_file_exists()

    def _ensure_file_exists(self) -> None:  # [추가] 누락 메서드 스텁
        if not self.tasks_file.exists():
            self.tasks_file.write_text("[]")

    def _load_tasks(self):  # [추가] 누락 메서드 스텁
        import json
        return json.loads(self.tasks_file.read_text())

    def _dict_to_task(self, data):  # [추가] 누락 메서드 스텁
        pass

    def get(self, task_id: UUID) -> Task:
        """ID로 작업 조회 - JSON 파일에서 검색"""
        tasks = self._load_tasks()
        for task_data in tasks:
            if UUID(task_data["id"]) == task_id:
                return self._dict_to_task(task_data)
        raise TaskNotFoundError(task_id)

    def save(self, task: Task) -> None:
        """작업 저장 - JSON 파일에 직렬화하여 저장"""
        # ... 나머지 구현
        pass


# 어떤 리포지토리든 동일한 인터페이스로 작동하는 것을 보여주는 예시
print("# 어떤 리포지토리든 동일하게 작동:")
print("# task = repository.get(task_id)")
print("# task.complete()")
print("# repository.save(task)")

### 06_config.py

## 구성 관리와 팩토리 패턴

이제 실제 리포지토리 인스턴스화를 처리하는 팩토리 구현 내에서 이 구성 기능을 활용해 보자. 애플리케이션 구성을 논의할 때 언급한 이 팩토리 패턴은 리포지토리 인스턴스를 적절히 구성해 깔끔하게 생성할 수 있다.

> **[노트북 참고]** 아래 셀은 노트북 환경에서 실행 가능하도록 `RepositoryType`, `os` 등 필요한 import가 추가되었습니다.

In [ ]:
# 구성 관리와 팩토리 패턴 - 환경 변수 기반 리포지토리 선택
# 팩토리 함수로 리포지토리 인스턴스를 적절히 구성하여 깔끔하게 생성
# [추가] 필요한 import
import os
from typing import Tuple
from todo_app.infrastructure.config import RepositoryType  # [추가]
from todo_app.application.repositories.task_repository import TaskRepository  # [추가]
from todo_app.application.repositories.project_repository import ProjectRepository  # [추가]
from todo_app.infrastructure.persistence.file import FileTaskRepository, FileProjectRepository  # [추가]
from todo_app.infrastructure.persistence.memory import InMemoryTaskRepository, InMemoryProjectRepository  # [추가]

# 환경 변수에서 리포지토리 타입을 읽어오는 구성 클래스
class Config:
    DEFAULT_REPOSITORY_TYPE = RepositoryType.MEMORY

    @classmethod
    def get_repository_type(cls) -> RepositoryType:
        repo_type_str = os.getenv("TODO_REPOSITORY_TYPE", cls.DEFAULT_REPOSITORY_TYPE.value)
        try:
            return RepositoryType(repo_type_str.lower())
        except ValueError:
            raise ValueError(f"잘못된 리포지토리 타입: {repo_type_str}")

    @classmethod
    def get_data_directory(cls):  # [추가] 시연에 필요한 메서드
        from pathlib import Path
        data_dir = os.getenv("TODO_DATA_DIR", "repo_data")
        path = Path(data_dir)
        path.mkdir(parents=True, exist_ok=True)
        return path


# 리포지토리 팩토리 함수 - 구성에 따라 적절한 리포지토리 구현체 생성
# 컴포지션 루트에서 호출되어 의존성 조립에 사용
def create_repositories() -> Tuple[TaskRepository, ProjectRepository]:
    repo_type = Config.get_repository_type()
    # 파일 기반 리포지토리 생성
    if repo_type == RepositoryType.FILE:
        data_dir = Config.get_data_directory()
        task_repo = FileTaskRepository(data_dir)
        project_repo = FileProjectRepository(data_dir)
        project_repo.set_task_repository(task_repo)
        return task_repo, project_repo
    # 인메모리 리포지토리 생성 (기본값)
    elif repo_type == RepositoryType.MEMORY:
        task_repo = InMemoryTaskRepository()
        project_repo = InMemoryProjectRepository()
        project_repo.set_task_repository(task_repo)
        return task_repo, project_repo
    else:
        raise ValueError(f"잘못된 리포지토리 타입: {repo_type}")

### 07_sendgrid_notifier.py

## SendGrid 외부 서비스 통합

애플리케이션 계층이 정의한 `NotificationPort` 인터페이스를 SendGrid로 구현하는 어댑터이다. 핵심 애플리케이션은 알림의 구체적 전송 방식(이메일, SMS 등)을 알 필요가 없다.

> **[노트북 참고]** 아래 셀은 노트북 환경에서 실행 가능하도록 `Task` import 추가, bare `except` 구문 수정, `TaskRepository` import 추가 등이 적용되었습니다.

In [ ]:
# SendGrid 외부 서비스 통합 - 포트/어댑터 패턴을 통한 외부 서비스 연결
# 애플리케이션 계층의 NotificationPort를 구현하는 인프라 계층 어댑터
from abc import ABC, abstractmethod
from dataclasses import dataclass
import os
from todo_app.domain.entities.task import Task  # [추가]
from todo_app.application.repositories.task_repository import TaskRepository  # [추가]
from todo_app.application.common.result import Result  # [추가]
from todo_app.application.dtos.task_dtos import CompleteTaskRequest  # [추가]

# 알림 포트(Port) - 애플리케이션 계층에서 정의하는 추상 인터페이스
class NotificationPort(ABC):
    """작업 이벤트에 대한 알림 전송 인터페이스"""

    @abstractmethod
    def notify_task_completed(self, task: Task) -> None:
        """작업이 완료되었을 때 알린다."""
        pass

    @abstractmethod
    def notify_task_high_priority(self, task: Task) -> None:
        """작업이 높은 우선순위로 설정되었을 때 알림"""
        pass

# 환경 변수에서 SendGrid 설정을 읽어오는 구성 클래스
class Config:
    """애플리케이션 구성"""

    @classmethod
    def get_sendgrid_api_key(cls) -> str:
        """SendGrid API 키 반환"""
        return os.getenv("TODO_SENDGRID_API_KEY", "")

    @classmethod
    def get_notification_email(cls) -> str:
        """알림 수신자 이메일 반환"""
        return os.getenv("TODO_NOTIFICATION_EMAIL", "")

# SendGrid 어댑터 - NotificationPort 인터페이스의 구체적 구현
# 외부 이메일 서비스의 세부사항을 캡슐화하는 인프라 계층 어댑터
class SendGridNotifier(NotificationPort):

    def __init__(self) -> None:
        self.api_key = Config.get_sendgrid_api_key()
        self.notification_email = Config.get_notification_email()
        self._init_sg_client()

    def _init_sg_client(self):  # [추가] 누락 메서드 스텁
        self.client = None

    # 이메일 알림 전송 - 설정이 없으면 조용히 건너뜀 (비즈니스 로직 방해 방지)
    def notify_task_completed(self, task: Task) -> None:
        """설정된 경우 완료된 작업에 대한 이메일 알림을 전송"""
        if not (self.client and self.notification_email):
            return
        try:
            print(f"[시연] 작업 완료 알림 전송: {task.title}")
        except Exception as e:  # [수정] bare except → except Exception as e
            pass  # 비즈니스 작업을 방해하지 않으면서 오류를 기록

    def notify_task_high_priority(self, task: Task) -> None:  # [추가] 추상 메서드 구현
        pass

# 유스케이스에서 포트를 통한 알림 연동 - 구현체가 무엇인지 알 필요 없음
@dataclass
class CompleteTaskUseCase:
    task_repository: TaskRepository
    notification_service: NotificationPort  # 포트에만 의존

    def execute(self, request: CompleteTaskRequest) -> Result:
        try:
            task = self.task_repository.get(request.task_id)
            task.complete(notes=request.completion_notes)
            self.task_repository.save(task)
            # 알림 전송 - SendGrid든 다른 서비스든 포트 인터페이스로 추상화
            self.notification_service.notify_task_completed(task)
        except Exception:  # [수정] bare except → except Exception
            pass

### 08_app_bootstrap.py

## 애플리케이션 부트스트래핑 (컴포지션 루트)

구성 요소 조율 논의에서 ﻿살펴본 것과 같이, 컴포지션 루트는 프레임워크 및 드라이버 계층의 모든 구성 요소를 통합하면서도 클린 아키텍처 경계를 유지한다. 이제 Application 컨테이너 클래스부터 시작하여 이 컴포지션의 구현을 더 자세히 살펴본다.

그런 다음 구현에서 __post_init__ 메서드를 사용하여 구성 요소를 생성한다.

> **[노트북 참고]** 아래 셀은 노트북 환경에서 실행 가능하도록 `dataclass`, `TaskRepository`, `NotificationPort`, `TaskPresenter` 등 필요한 import가 모두 추가되었습니다.

In [ ]:
# 컴포지션 루트(Composition Root) - 모든 의존성을 조립하는 유일한 장소
# Application 컨테이너 + 팩토리 함수 + main 진입점으로 구성
# [추가] 필요한 import
import sys
from dataclasses import dataclass
from todo_app.application.repositories.task_repository import TaskRepository
from todo_app.application.repositories.project_repository import ProjectRepository
from todo_app.application.service_ports.notifications import NotificationPort
from todo_app.interfaces.presenters.base import TaskPresenter, ProjectPresenter
from todo_app.application.use_cases.task_use_cases import (
    CreateTaskUseCase, CompleteTaskUseCase, GetTaskUseCase,
    DeleteTaskUseCase, UpdateTaskUseCase,
)
from todo_app.interfaces.controllers.task_controller import TaskController
from todo_app.infrastructure.repository_factory import create_repositories
try:  # [수정] sendgrid 미설치 시에도 동작하도록 보호
    from todo_app.infrastructure.notifications.factory import create_notification_service
except (ImportError, ModuleNotFoundError):
    pass
from todo_app.infrastructure.notifications.recorder import NotificationRecorder
from todo_app.interfaces.presenters.cli import CliTaskPresenter, CliProjectPresenter
try:  # [수정] click 미설치 시에도 동작하도록 보호
    from todo_app.infrastructure.cli.click_cli_app import ClickCli as _RealClickCli
except (ImportError, ModuleNotFoundError):
    pass

# [추가] sendgrid 미설치 시 팩토리 함수 대체 정의
if 'create_notification_service' not in dir():
    def create_notification_service() -> NotificationPort:
        """[스텁] sendgrid 미설치 시 NotificationRecorder로 폴백"""
        return NotificationRecorder()

# Application 컨테이너 - 모든 구성 요소 간의 관계를 정의하는 중앙 컨테이너
# 의존성 주입을 통해 각 계층의 구성 요소를 연결
@dataclass
class Application:
    """모든 구성 요소를 연결하는 컨테이너."""

    # 외부에서 주입되는 의존성들 (인터페이스 타입으로 선언)
    task_repository: TaskRepository
    project_repository: ProjectRepository
    notification_service: NotificationPort
    task_presenter: TaskPresenter
    project_presenter: ProjectPresenter

    # 의존성 조립: 유스케이스와 컨트롤러를 생성하고 연결
    def __post_init__(self):
        """유스 케이스와 컨트롤러 연결"""
        # 유스케이스 생성 - 리포지토리와 서비스 포트를 주입
        self.create_task_use_case = CreateTaskUseCase(self.task_repository, self.project_repository)
        self.complete_task_use_case = CompleteTaskUseCase(
            self.task_repository, self.notification_service
        )
        self.get_task_use_case = GetTaskUseCase(self.task_repository)
        self.delete_task_use_case = DeleteTaskUseCase(self.task_repository)
        self.update_task_use_case = UpdateTaskUseCase(
            self.task_repository, self.notification_service
        )
        # 컨트롤러 생성 - 유스케이스와 프레젠터를 주입
        self.task_controller = TaskController(
            create_use_case=self.create_task_use_case,
            complete_use_case=self.complete_task_use_case,
            update_use_case=self.update_task_use_case,
            delete_use_case=self.delete_task_use_case,
            get_use_case=self.get_task_use_case,
            presenter=self.task_presenter,
        )
        # ... 프로젝트 유스 케이스와 컨트롤러 생성


# 팩토리 함수 - Application 컨테이너에 주입할 인스턴스를 구성하고 생성
def create_application(
    notification_service: NotificationPort,
    task_presenter: TaskPresenter,
    project_presenter: ProjectPresenter,
) -> "Application":
    """Application 컨테이너를 위한 팩토리 함수"""
    task_repository, project_repository = create_repositories()
    notification_service = create_notification_service()
    return Application(
        task_repository=task_repository,
        project_repository=project_repository,
        notification_service=notification_service,
        task_presenter=task_presenter,
        project_presenter=project_presenter,
    )


# main 진입점 - 컴포지션 루트의 최상위, 모든 구성 요소가 인스턴스화되는 유일한 장소
def main() -> int:
    """CLI 애플리케이션의 메인 진입점"""
    try:
        # 의존성 조립 후 Application 생성
        app = create_application(
            notification_service=NotificationRecorder(),
            task_presenter=CliTaskPresenter(),
            project_presenter=CliProjectPresenter(),
        )

        # CLI 프레임워크 어댑터 생성 및 실행
        cli = ClickCli(app)
        return cli.run()

    except KeyboardInterrupt:
        print("\n안녕히 가세요!")
        return 0
    except Exception as e:
        print(f"오류: {str(e)}", file=sys.stderr)
        return 1


# [보완] 노트북 환경에서는 main()을 직접 실행하지 않습니다.
print("main() 함수가 정의되었습니다. CLI를 실행하려면 main()을 호출하세요.")